# Criba de motores TTS — XTTS v2 (base y tu fine-tune navalchris)

La referencia de lo que ya tienes. Dos variantes: **XTTS v2 base** en zero-shot con las 55 frases, y tu **fine-tune `xtts_finetunned_navalchris`** solo con las de Naval (`n` y `k`), que nunca llegó a medirse. Mismos parámetros que tu cuaderno de producción: `temperature=0.75`, `repetition_penalty=3.0`, `top_p=0.8`, `top_k=60`.

**GPU:** T4. **Tiempo estimado:** unos 40 min, incluida la instalación.

**Cómo se usa** (es igual en todos los cuadernos de la criba):

1. *Entorno de ejecución → Cambiar tipo de entorno* → **T4**.
2. Ejecuta la **celda 1** (instalación). Cuando termine, *Entorno de ejecución → Reiniciar sesión*.
3. Ejecuta las demás celdas **en orden**, de la 2 a la última. La celda 2 comprueba que estén las
   referencias y las frases antes de cargar ningún modelo: si dice `RESULTADO: REVISAR`, para ahí.
4. La última celda **desconecta la GPU sola** para que no siga gastando unidades.

Si la sesión se corta a medias, vuelve a empezar por la celda 1: lo que ya esté en Drive no se
genera otra vez.

Salida: `fine_tunned_naval/outputs/criba/<variante>/` en Drive, un `.wav` por item (`n01`, `v01`,
`b01`, `k01`…) más `registro.jsonl`, `parametros.json` y `entorno.json`.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
# Las mismas versiones con las que se entrenó navalchris.
!pip install -q coqui-tts==0.27.5 "coqui-tts-trainer>=0.3.0,<0.4.0" "transformers==4.57.1"

In [ ]:
# 2. Preparar: código del repo, Drive, frases y referencias
import os, sys, subprocess
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun

print(sys.version.split()[0], "|", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                                  capture_output=True, text=True).stdout.strip())
comun.montar_drive()
ITEMS, SILENCIO_MS = comun.cargar_items(None)
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Funciones de carga y síntesis
import os, torch
os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.api import TTS
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

PESOS = comun.DRIVE / "weights"
GEN = dict(temperature=0.75, repetition_penalty=3.0, top_p=0.8, top_k=60, length_penalty=1.0,
           enable_text_splitting=False)

def cargar(variante):
    if variante == "xtts_base":
        return TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda").synthesizer.tts_model
    carpeta = PESOS / "xtts_finetunned_navalchris"
    config = XttsConfig(); config.load_json(str(carpeta / "config.json"))
    m = Xtts.init_from_config(config)
    m.load_checkpoint(config, checkpoint_path=str(carpeta / "best_model.pth"),
                      vocab_path=str(PESOS / "vocab.json"), use_deepspeed=False)
    return m.cuda()

def hacer_sintetizar(modelo):
    latentes = {}
    def sintetizar(texto, ref):
        if ref["wav"] not in latentes:
            latentes[ref["wav"]] = modelo.get_conditioning_latents(
                audio_path=[ref["wav"]], gpt_cond_len=12, gpt_cond_chunk_len=6, max_ref_length=12)
        gpt, spk = latentes[ref["wav"]]
        out = modelo.inference(texto, "es", gpt, spk, **GEN)
        return out["wav"], 24000
    return sintetizar
print("OK")

In [ ]:
# 4. Generar las dos variantes
SOLO_NAVAL = [it for it in ITEMS if it["voz"] == "naval"]
for variante, items in [("xtts_base", ITEMS), ("xtts_navalchris", SOLO_NAVAL)]:
    print(f"\n=== {variante} ({len(items)} items)")
    try:
        modelo = cargar(variante)
    except Exception as e:
        print(f"No se pudo cargar {variante}: {type(e).__name__}: {e}. Paso a la siguiente.")
        continue
    comun.guardar_entorno(variante, ["coqui-tts", "torch", "transformers"])
    comun.generar(variante, hacer_sintetizar(modelo), items, REFS, SILENCIO_MS, params=GEN)
    del modelo
    comun.liberar()

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
for variante in ['xtts_base', 'xtts_navalchris']:
    reg = comun.SALIDA / variante / "registro.jsonl"
    if not reg.exists():
        print(f"{variante}: sin registro")
        continue
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines()]
    ok = sum(f["ok"] for f in filas)
    print(f"{variante}: {ok} items correctos, {len(filas) - ok} con error")
comun.desconectar()